# 03 · Map a county: inference → merge → export
Each stage saves to the run folder. If Colab disconnects, re-run from the top — finished rows are skipped.

In [ ]:
# --- SETUP (same in every notebook) -----------------------------------
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys, subprocess
REPO = "/content/pei-landuse"
GITHUB = "github.com/YOUR_USER/pei-landuse.git"     # private repo
token = userdata.get("GITHUB_TOKEN")                 # Colab ▸ Secrets (key icon). Never paste tokens in cells.
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", f"https://{token}@{GITHUB}", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-colab.txt"], check=True)
sys.path.insert(0, f"{REPO}/src")
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
CONFIG = "configs/infer_2020_prince.yaml"

import torch, rasterio
from pei.config import load_config, make_run_dir
from pei.model import load_for_inference
from pei.infer import run_inference, load_raw_detections
from pei.merge import merge_detections
from pei.export import finalize

cfg = load_config(CONFIG)
run_dir = make_run_dir(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("run folder:", run_dir)

## Stage 1 · Inference (GPU, resumable)

In [ ]:
model = load_for_inference(cfg, device)
run_inference(cfg, model, run_dir, device)

## Stage 2 · Merge (CPU). Re-runnable on its own with different `merge:` settings — no need to redo inference.

In [ ]:
raw = load_raw_detections(run_dir)
with rasterio.open(cfg["paths"]["input_raster"]) as src:
    pixel_size = (src.transform[0], -src.transform[4])
fields = merge_detections(cfg, raw, run_dir, pixel_size)

## Stage 3 · Export GeoPackage for ArcGIS

In [ ]:
out = finalize(cfg, fields, run_dir)
print(out)